# Imports

In [61]:
"""
val.lmdb keys: 0 - 67520

"""

'\nval.lmdb keys: 0 - 67520\n\n'

In [5]:
import os
import pickle as pkl
from pathlib import Path
from typing import Tuple
from tqdm import tqdm
import lmdb
import json
import torch
from ase.atoms import Atoms
from ase.data import chemical_symbols
from dotenv import load_dotenv
from optimade.client import OptimadeClient
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.core import Structure
from mp_api.client import MPRester
from colabfit.tools.vast.database import batched
from colabfit.tools.vast.configuration import AtomicConfiguration

load_dotenv()
mp_key = os.getenv("MP_KEY")

In [63]:
from colabfit.tools.vast.database import generate_ds_id

generate_ds_id()
# DS_5rjlk0wubpsf_0 test
# DS_uluw9723f2n4_0 train
# DS_2dgg8tui3p9x_0 val

Generated new DS ID: DS_c3f6bsqpbro5_0


'DS_c3f6bsqpbro5_0'

## Get AtomicConfigurations from pickles

In [64]:
with open("alex_mp_pickles/materials_project_ids_test_0.pickle", "rb") as f:
    configs = pkl.load(f)

In [65]:
for config in configs:
    config.info["_name"] = (
        f"alex_mp_{config.info['alex_mp_split']}_{config.info['id'].replace('alex<', '').replace('>', '')}"
    )

In [66]:
x = AtomicConfiguration.from_ase(configs[0])
x.info

{'alex_mp_split': 'test',
 'alex_mp_file_key': '135647',
 'space_group': 'P6/mmm',
 'chemical_system': 'B-Ca-Pt',
 'energy_above_hull': 0.035818815333334,
 'dft_band_gap': nan,
 'dft_bulk_modulus': nan,
 'dft_mag_density': 7.228765166347972e-05,
 'hhi_score': 8500.524569238582,
 'ml_bulk_modulus': 153.05267333984375,
 'id': 'alex<agm003037050>',
 'energy': -60.6122184,
 'forces': [[-0.00021621, -5.875e-05, 2.112e-05],
  [-0.00023181, 8.608e-05, 6.16e-06],
  [-2.82e-05, -1.06e-06, 0.00019908],
  [0.00031906, -1.347e-05, 0.00018031],
  [0.00041832, 0.00116541, 0.01077765],
  [-0.00029071, -0.0012233, -0.01089106],
  [0.00033886, -0.00057788, -0.01084804],
  [-0.00032143, 0.00061737, 0.01071666],
  [0.00025886, -0.00137379, -0.01094846],
  [-0.00024672, 0.00137941, 0.01078657]],
 'stress': [[-2.9770522, -0.02345597, -0.03211148],
  [-0.02345609, -2.9897566, -0.01380648],
  [-0.03211154, -0.01380642, 7.3631196]],
 'alexandria_source': 'pbe',
 '_name': 'alex_mp_test_agm003037050'}

# Functions

In [7]:
def from_lmdb(
    env: lmdb.Environment,
    key: str,
    split_name: str,
):
    """
    Read ase.atoms.Atoms object from lmdb.

    Args:
        env: lmdb.Environment object.
        key: key to read the configuration from the lmdb.
    """
    with env.begin() as txn:
        data = txn.get(key.encode())
        if data is None:
            raise ValueError(f"Key {key} not found in the lmdb.")
        lmdb_config = pkl.loads(data)
        cell = lmdb_config.pop("cell", None)
        numbers = lmdb_config.pop("atomic_numbers", None)
        coords = lmdb_config.pop("pos", None)
        if coords is None:
            raise ValueError(f"Key {key} has no coordinates in the lmdb.")
        pbc = lmdb_config.pop("pbc", None)
        # ds_idx = lmdb_config.get("ds_idx", 0)
        identifier = key
        property_keys = lmdb_config.keys()
        property_dict = {}
        if isinstance(property_keys, str):
            property_keys = (property_keys,)
        for prop in property_keys:
            property_dict[prop] = lmdb_config.get(prop, None)
            if (
                property_dict[prop] is not None
                and torch.is_tensor(property_dict[prop])
                and property_dict[prop].is_floating_point()
            ):
                property_dict[prop] = property_dict[prop]
        pbc = [bool(i) for i in pbc]
        config = Atoms(
            numbers=numbers,
            cell=cell,
            positions=coords,
            pbc=pbc,
        )
        config.info["alex_mp_split"] = split_name
        config.info["alex_mp_file_key"] = identifier
        config.info.update(property_dict)
        config.info["id"] = config.info.pop("ids", None)
    return config


def read_lmdb(
    lmdb_path: str,
):
    """
    Read ase.atoms.Atoms object from lmdb.

    Args:
        lmdb_path: Path to the lmdb database.
        key: key to read the configuration from the lmdb.
        split_name: name of the split (e.g. 'train', 'val', 'test').
        property_keys: list of property keys to read from the lmdb.
    """
    if isinstance(lmdb_path, str):
        lmdb_path = Path(lmdb_path)
    env = lmdb.open(
        str(lmdb_path),
        readonly=True,
        lock=False,
        subdir=False,
        readahead=False,
        meminit=False,
    )

    try:
        keys = set()
        with env.begin() as txn:
            cursor = txn.cursor()
            for key, value in cursor:
                keys.add(key.decode())
        for key in keys:
            config = from_lmdb(
                env=env,
                key=key,
                split_name=lmdb_path.stem,
            )
            yield config
    finally:
        env.close()

# Create config generator and data variables

In [8]:
config_gen = read_lmdb("/home/gpwolfe/colabfit/vast_ingest/alex_mp/data/test.lmdb")

In [9]:
x = next(config_gen)
x.info

{'alex_mp_split': 'test',
 'alex_mp_file_key': '322447',
 'space_group': 'P-4m2',
 'chemical_system': 'Ge-Li-Ni-Sm',
 'energy_above_hull': 0.0747718416176503,
 'dft_band_gap': nan,
 'dft_bulk_modulus': nan,
 'dft_mag_density': 9.882978543240125e-08,
 'hhi_score': 2295.70676114153,
 'ml_bulk_modulus': 94.23027801513672,
 'id': 'alex<agm004510071>'}

# Get all data

## Get materials project ids

In [69]:
lmdb_path = Path("/home/gpwolfe/colabfit/vast_ingest/alex_mp/data/test.lmdb")
env = lmdb.open(
    str(lmdb_path),
    readonly=True,
    lock=False,
    subdir=False,
    readahead=False,
    meminit=False,
)
keys = set()
with env.begin() as txn:
    cursor = txn.cursor()
    for key, value in cursor:
        keys.add(key.decode())

In [70]:
mp_ids = []
for key in keys:
    with env.begin() as txn:
        data = txn.get(key.encode())
        lmdb_config = pkl.loads(data)
        if "mp" in lmdb_config["ids"]:
            mp_ids.append(lmdb_config["ids"])

KeyboardInterrupt: 

In [ ]:
matid_taskid = []
for x in data:
    if x.entries.R2SCAN:
        matid_taskid.append((x.material_id, x.entries.R2SCAN.data["task_id"], "R2SCAN"))
    elif x.entries.GGA_U:
        matid_taskid.append((x.material_id, x.entries.GGA_U.data["task_id"], "GGA_U"))
    elif x.entries.GGA:
        matid_taskid.append((x.material_id, x.entries.GGA.data["task_id"], "GGA"))
    else:
        print(f"No valid entry found for material {x.material_id}")

# Testing

In [77]:
config_gen = read_lmdb("/home/gpwolfe/colabfit/vast_ingest/alex_mp/data/test.lmdb")
for i, conf in tqdm(enumerate(config_gen)):
    with open("materials_project_ids_test.csv", "a") as f:
        if "mp-" in conf.info["id"]:
            f.write(f"{conf.info['id']}\n")

67521it [00:19, 3485.81it/s]


In [3]:
mpids = Path("materials_project_ids_train.csv").read_text().splitlines()
# mpids = Path("materials_project_ids_val.csv").read_text().splitlines()
# mpids = Path("materials_project_ids_test.csv").read_text().splitlines()

print(len(mpids))

35255


In [5]:
with MPRester(api_key=mp_key) as mpr:

    for batch in batched(list(set(mpids)), 4000):
        # actual_ids = set(batch)
        data = mpr.materials.search(material_ids=batch)
        # returned_ids = set(str(x.material_id) for x in data)
        # missing_mpids = list(actual_ids - returned_ids)
        # print(len(missing_mpids))
        # with open("missing_mpids.csv", "a") as f:
        #     for x in missing_mpids:
        #         f.write(f"{x}\n")
        tasks_to_write = []
        for material in data:
            tasks = material.run_types
            mat_id = str(material.material_id)
            priority = -1
            for task_id, task_type in tasks.items():
                task_id = str(task_id)
                if str(task_type) == "GGA":
                    if priority >= 0:
                        continue
                    t_type = str(task_type)
                    priority = 0
                    t_type = str(task_type)
                if str(task_type) == "GGA+U":
                    if priority >= 1:
                        continue
                    t_type = str(task_type)
                    priority = 1
                if str(task_type) == "r2SCAN":
                    t_type = str(task_type)
                    priority = 2
            if priority == -1:
                print(f"Material {mat_id} has no valid task type")
            tasks_to_write.append(f"{mat_id},{task_id},{t_type}")

        with open("material_to_task_map.csv", "a") as f:
            for x in tasks_to_write:
                f.write(f"{x}\n")

Retrieving MaterialsDoc documents:   0%|          | 0/3924 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3927 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3918 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3914 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3915 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3924 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3940 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3924 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/3203 [00:00<?, ?it/s]

In [ ]:
with open("material_to_task_map_train.csv") as f:
    mp_map = {}
    for line in f:
        mat, task, method = line.strip().split(",")
        if not mp_map.get(mat):
            mp_map[mat] = {}
            mp_map[mat][task] = method
mp_tasks = {}
for fp in Path().glob("materials_project_tasks_train*.json"):
    with open(fp) as f:
        tasks = json.load(f)
        print(len(tasks))
        mp_tasks.update(tasks)

5000
5000
5000
5000
5000
4589
5000
255


In [ ]:
split = "val"
config_gen = read_lmdb(f"/home/gpwolfe/colabfit/vast_ingest/alex_mp/data/{split}.lmdb")


# do something with the mapping to get the type of mp task and the task id
def get_alex_batch(alex_id_batch):
    alex_missing = set()
    pbe_dict = {}
    pbesol_dict = {}
    for sub_batch in batched(alex_id_batch, 100):
        alex_client = OptimadeClient(
            [
                "https://alexandria.icams.rub.de/pbe",
                "https://alexandria.icams.rub.de/pbesol",
            ],
        )
        alex_id_filter = 'id="' + '" OR id="'.join(sub_batch) + '"'
        alex_response = alex_client.get(filter=alex_id_filter)
        key1 = list(alex_response["structures"].keys())[0]
        pbe = list(alex_response["structures"][key1].keys())[0]
        pbesol = list(alex_response["structures"][key1].keys())[1]
        print(pbesol)
        print(key1)
        print(alex_response["structures"].keys())
        print(alex_response["structures"][key1].keys())
        pbesol_ids = [
            x["id"] for x in alex_response["structures"][key1][pbesol]["data"]
        ]
        pbe_ids = [
            x["id"]
            for x in alex_response["structures"][key1][pbe]["data"]
            if x["id"] not in pbesol_ids
        ]
        alex_missing_batch = set(alex_id_batch) - (set(pbe_ids + pbesol_ids))
        pbe_dict_batch = {
            x["id"]: {
                "energy": x["attributes"]["_alexandria_energy"],
                "forces": x["attributes"]["_alexandria_forces"],
                "stress": x["attributes"]["_alexandria_stress_tensor"],
            }
            for x in alex_response["structures"][key1][pbe]["data"]
        }
        pbesol_dict_batch = {
            x["id"]: {
                "energy": x["attributes"]["_alexandria_energy"],
                "forces": x["attributes"]["_alexandria_forces"],
                "stress": x["attributes"]["_alexandria_stress_tensor"],
            }
            for x in alex_response["structures"][key1][pbesol]["data"]
        }
        pbe_dict.update(pbe_dict_batch)
        pbesol_dict.update(pbesol_dict_batch)
        alex_missing.update(alex_missing_batch)
    return pbe_dict, pbesol_dict, alex_missing


for batch_num, batch in enumerate(batched(config_gen, 10000)):
    configs = []
    alex_ids = [
        conf.info["id"].replace("alex<", "")[:-1]
        for conf in batch
        if "alex" in conf.info["id"]
    ]
    mp_ids = [conf.info["id"] for conf in batch if "mp-" in conf.info["id"]]
    pbe_dict, pbesol_dict, alex_missing = get_alex_batch(alex_ids)
    for i, conf in tqdm(enumerate(batch)):
        if "alex" in conf.info["id"]:
            alex_id = conf.info["id"].replace("alex<", "")[:-1]
            if alex_id in pbe_dict:
                conf.info.update(pbe_dict[alex_id])
            elif alex_id in pbesol_dict:
                conf.info.update(pbesol_dict[alex_id])
            else:
                conf.info["missing_alexandria_object"] = True
        if "mp-" in conf.info["id"]:
            mp_id = conf.info["id"]
            try:
                task_id = list(mp_map[mp_id].keys())[0]
                task = mp_tasks[task_id]
                conf.info.update(task)
            except KeyError:
                conf.info["missing_mp_task"] = True
        configs.append(conf)
    with open(f"materials_project_ids_{split}_{batch_num}.pickle", "wb") as f:
        pkl.dump(configs, f)
    break

In [3]:
pickled_configs = Path(
    f"alex_mp_pickles3/materials_project_ids_train_15.pickle"
).read_bytes()
configs = pkl.loads(pickled_configs)

In [4]:
for config in configs:
    if "mp-" in config.info["id"]:
        print(config.info)
        break

{'alex_mp_split': 'train', 'alex_mp_file_key': '168260', 'space_group': 'I-42d', 'chemical_system': 'Gd-Mn-S', 'energy_above_hull': 0.0821659300000021, 'dft_band_gap': 0.8582999999999998, 'dft_bulk_modulus': nan, 'dft_mag_density': 0.1310185472976075, 'hhi_score': 2415.3168634608564, 'ml_bulk_modulus': 52.14632797241211, 'id': 'mp-674348', 'materials_project_id': 'mp-674348', 'forces': [[0.0, 0.05785371, 0.0], [-0.05785371, -0.0, 0.0], [0.0, -0.05785371, 0.0], [0.05785371, -0.0, 0.0], [0.0, -0.0, 0.0], [0.0, -0.0, 0.0], [-0.04539895, 0.03356165, 0.05068496], [-0.03356165, -0.04539895, -0.05068496], [0.03356165, -0.04539895, 0.05068496], [-0.04539895, -0.03356165, -0.05068496], [-0.03356165, 0.04539895, 0.05068496], [0.04539895, 0.03356165, -0.05068496], [0.04539895, -0.03356165, 0.05068496], [0.03356165, 0.04539895, -0.05068496]], 'energy': -129.558266, 'stress': [[-2.75390807, 0.0, 0.0], [0.0, -2.75390807, 0.0], [-0.0, -0.0, -2.37348669]], 'mp_task_id': 'mp-1807360', 'mp_run_type': 'G

In [ ]:
with MPRester(mp_key) as mpr:
    data = mpr.materials.search(material_ids=["mp-1018671"])

Retrieving MaterialsDoc documents: 100%|██████████| 1/1 [00:00<00:00, 23301.69it/s]


### Check missing/empty ids

In [ ]:
with open("empty_mp_test_ids.csv", "r") as f:
    empty_mp_ids = f.read().splitlines()
mp_ids = set([x for x in empty_mp_ids if "alex" not in x])
print(len(mp_ids))

3831


In [ ]:
with MPRester(api_key=mp_key) as mpr:
    data = mpr.materials.search(material_ids=mp_ids)

Retrieving MaterialsDoc documents:   0%|          | 0/3756 [00:00<?, ?it/s]

In [ ]:
data[0].entries.R2SCAN

mp-1025172-r2SCAN ComputedStructureEntry - Ho1 Ti2 Ga4  (Ho(TiGa2)2)
Energy (Uncorrected)     = -108.7077 eV (-15.5297 eV/atom)
Correction               = 0.0000    eV (0.0000   eV/atom)
Energy (Final)           = -108.7077 eV (-15.5297 eV/atom)
Energy Adjustments:
  None
Parameters:
  potcar_spec            = [{'titel': 'PAW_PBE Ho_3 06Sep2000', 'hash': '335fa7a1baca529b5e31c04b797b8f6f', 'summary_stats': None}, {'titel': 'PAW_PBE Ti_pv 07Sep2000', 'hash': 'cdd047e254a1247dbf5c1bb0563a75b9', 'summary_stats': None}, {'titel': 'PAW_PBE Ga_d 06Jul2010', 'hash': '7bb23032e63255efc79cad079baff5aa', 'summary_stats': None}]
  run_type               = r2SCAN
  is_hubbard             = False
  hubbards               = None
Data:
  oxide_type             = None
  aspherical             = True
  last_updated           = 2024-11-21 20:52:48.026419+00:00
  task_id                = mp-2046286
  material_id            = mp-1025172

In [ ]:
tasks_to_write = []
for material in data:
    tasks = material.run_types
    mat_id = str(material.material_id)
    priority = 0
    for task_id, task_type in tasks.items():
        task_id = str(task_id)
        if str(task_type) == "GGA":
            if priority > 0:
                continue
            t_type = str(task_type)
        if str(task_type) == "GGA_U":
            if priority > 1:
                continue
            t_type = str(task_type)
            priority = 1
        if str(task_type) == "r2SCAN":
            t_type = str(task_type)
            priority = 2
    tasks_to_write.append(f"{mat_id},{task_id},{t_type}")
with open("material_to_task_map_train3.csv", "w") as f:
    for x in tasks_to_write:
        f.write(f"{x}\n")

In [ ]:
with MPRester(api_key=mp_key) as mpr:
    task_data = mpr.materials.tasks.search(["mp-2046286"])

Retrieving TaskDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
task_data[0].output.stress

[[0.43611279000000003, -0.0, -7e-08],
 [0.0, 0.43611279000000003, 0.0],
 [-7e-08, -0.0, 1.13490461]]

In [ ]:
with MPRester(api_key=mp_key) as mpr:
    for batch in batched(list(set(mpids)), 4000):
        data = mpr.materials.search(material_ids=batch)
        tasks_to_write = []
        for material in data:
            tasks = material.run_types
            mat_id = str(material.material_id)
            priority = 0
            for task_id, task_type in tasks.items():
                task_id = str(task_id)
                if str(task_type) == "GGA":
                    if priority > 0:
                        continue
                    t_type = str(task_type)
                if str(task_type) == "GGA_U":
                    if priority > 1:
                        continue
                    t_type = str(task_type)
                    priority = 1
                if str(task_type) == "r2SCAN":
                    t_type = str(task_type)
                    priority = 2
            tasks_to_write.append(f"{mat_id},{task_id},{t_type}")
            break
        # with open("material_to_task_map_train3.csv", "a") as f:
        #     for x in tasks_to_write:
        #         f.write(f"{x}\n")

In [ ]:
with open("material_to_task_map_train.csv", "r") as f:
    print(f.readline())

mp-1009133,mp-2423708,GGA



In [ ]:
with open("material_to_task_map_train2.csv", "r") as f:
    missearch = [line.strip().split(",") for line in batch]
    missearch = {x[1]: x[0] for x in missearch}

### This is a corrupted version that was being used with materials.search. See below for materials.tasks.search

In [ ]:
all_results = {}
with open("material_to_task_map.csv", "r") as f:
    with MPRester(api_key=mp_key) as mp:
        for i, batch in enumerate(batched(f.readlines(), 5000)):
            task_batch = [line.strip().split(",")[1] for line in batch]
            task_mat_map = [line.strip().split(",") for line in batch]
            task_mat_map = {x[1]: x[0] for x in task_mat_map}
            # miss_mats = mp.materials.search(
            #     material_ids=[task_mat_map[x] for x in missing]
            # )
            miss_mats_results = {}
            for m in tasks:
                if m.entries.R2SCAN:
                    miss_mats_results[m.material_id] = {
                        "forces": None,
                        "energy": m.entries.R2SCAN.energy,
                        "stress": None,
                        "task_id": str(task.task_id),
                    }
                elif m.entries.GGA_U:
                    miss_mats_results[m.material_id] = {
                        "forces": None,
                        "energy": m.entries.GGA_U.energy,
                        "stress": None,
                        "task_id": str(task.task_id),
                    }
                elif m.entries.GGA:
                    miss_mats_results[m.material_id] = {
                        "forces": None,
                        "energy": m.entries.GGA.energy,
                        "stress": None,
                        "task_id": str(task.task_id),
                    }
                else:
                    print(f"No valid entry found for material {m.material_id}")
                    break
            # found = {
            #     str(task.task_id): {
            #         "forces": task.output.forces,
            #         "energy": task.output.energy,
            #         "stress": task.output.stress,
            #     }
            #     for task in tasks
            # }
            # found.update(miss_mats_results)
            all_results.update(miss_mats_results)
        # break
with open(f"materials_project_tasks.json", "w") as f:
    json.dump(
        all_results,
        f,
    )

### For materials.tasks.search

In [ ]:
task_results = {}
with open("material_to_task_map.csv", "r") as f:
    with MPRester(api_key=mp_key) as mp:
        for i, batch in enumerate(batched(f.readlines(), 5000)):
            task_batch = [line.strip().split(",")[1] for line in batch]
            task_mat_map = [line.strip().split(",") for line in batch]
            task_mat_map = {x[1]: x[0] for x in task_mat_map}
            tasks = mp.materials.tasks.search(task_ids=task_batch)
            for m in tasks:
                run_type = m.run_type
                material_id = task_mat_map[m.task_id]
                task_results[material_id] = {
                    "forces": m.output.forces,
                    "energy": m.output.energy,
                    "stress": m.output.stress,
                    "mp_task_id": str(m.task_id),
                    "mp_run_type": str(m.run_type),
                }

Retrieving TaskDoc documents:   0%|          | 0/4992 [00:00<?, ?it/s]

Retrieving TaskDoc documents:   0%|          | 0/4994 [00:00<?, ?it/s]

Retrieving TaskDoc documents:   0%|          | 0/4998 [00:00<?, ?it/s]

Retrieving TaskDoc documents:   0%|          | 0/4994 [00:00<?, ?it/s]

Retrieving TaskDoc documents:   0%|          | 0/4993 [00:00<?, ?it/s]

Retrieving TaskDoc documents:   0%|          | 0/4997 [00:00<?, ?it/s]

Retrieving TaskDoc documents:   0%|          | 0/4584 [00:00<?, ?it/s]

In [ ]:
# with open(f"materials_project_properties.json", "r") as f:
#     data = json.load(f)
#     data.update(task_results)
# with open(f"materials_project_properties.json", "w") as f:
#     json.dump(
#         data,
#         f,
#     )

In [ ]:
# with open(f"materials_project_properties.json", "w") as f:
#     json.dump(
#         task_results,
#         f,
#     )